In [1]:
import pandas as pd

# Đọc các file pickle từ thư mục ../data/
train_df = pd.read_pickle('../data/train_fe.pkl')
val_df = pd.read_pickle('../data/val_fe.pkl')
test_df = pd.read_pickle('../data/test_fe.pkl')

# Kiểm tra dữ liệu (ví dụ: in ra số dòng và cột)
print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")


Train shape: (1577139, 71)
Val shape: (337206, 71)
Test shape: (373699, 71)


In [2]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, SelectPercentile
from xgboost import XGBRegressor, XGBClassifier
import pandas as pd
import numpy as np
import json

# ════════════════════════════════════════════════════════════════
# 1. CHUẨN BỊ DỮ LIỆU
# ════════════════════════════════════════════════════════════════
DROP_COLS = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']

y_train = train_df['target_log_revenue']
X_train = train_df.drop(columns=DROP_COLS)

y_val = val_df['target_log_revenue']
X_val = val_df.drop(columns=DROP_COLS)

print(f"Số lượng đặc trưng ban đầu: {len(X_train.columns)}")

# ════════════════════════════════════════════════════════════════
# LỚP 1: VARIANCE THRESHOLD
# ════════════════════════════════════════════════════════════════
var_selector = VarianceThreshold(threshold=(.995 * (1 - .995)))
var_selector.fit(X_train)

X_train_v = X_train.loc[:, var_selector.get_support()]
X_val_v   = X_val.loc[:, var_selector.get_support()]
print(f"1. Sau Variance Threshold: Còn {X_train_v.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 2: CORRELATION FILTER
# ════════════════════════════════════════════════════════════════
corr_matrix = X_train_v.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

X_train_c = X_train_v.drop(columns=to_drop_corr)
X_val_c   = X_val_v.drop(columns=to_drop_corr)
print(f"2. Sau Correlation Filter: Còn {X_train_c.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 3: MUTUAL INFORMATION
# ════════════════════════════════════════════════════════════════
print("3. Đang thực hiện Mutual Information...")
mi_percentile = 95 if X_train_c.shape[1] < 100 else 90
mi_selector = SelectPercentile(mutual_info_regression, percentile=mi_percentile)

# fillna(-1) chỉ dùng trong MI (không ảnh hưởng X_train_c gốc)
mi_selector.fit(X_train_c.fillna(-1), y_train)  

X_train_mi = X_train_c.loc[:, mi_selector.get_support()]
X_val_mi   = X_val_c.loc[:, mi_selector.get_support()]
print(f"3. Sau Mutual Information (Top {mi_percentile}%): Còn {X_train_mi.shape[1]} biến")

# ════════════════════════════════════════════════════════════════
# LỚP 4: 2-STAGE XGBOOST FEATURE IMPORTANCE (Union Approach)
# ════════════════════════════════════════════════════════════════
print("4. Đang chạy XGBoost Importance cho cả 2 tầng (Classification + Regression)...")

# --- 4A. CLASSIFICATION IMPORTANCE (Ai sẽ mua?) ---
y_train_buy = (y_train > 0).astype(int)
y_val_buy   = (y_val > 0).astype(int)

model_fs_clf = XGBClassifier(
    n_estimators=500, max_depth=3, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='logloss'
)

model_fs_clf.fit(
    X_train_mi, y_train_buy,
    eval_set=[(X_val_mi, y_val_buy)],
    verbose=False
)

imp_clf = pd.DataFrame({
    'feature': X_train_mi.columns,
    'importance': model_fs_clf.feature_importances_
}).sort_values(by='importance', ascending=False)

# Cumulative 95% cho Classification
cutoff_clf = (imp_clf['importance'].cumsum() <= 0.95).sum()
keep_clf = imp_clf.iloc[:cutoff_clf + 1]['feature'].tolist()

# --- 4B. REGRESSION IMPORTANCE (Mua bao nhiêu? - Chỉ dùng tập Buyers) ---
mask_buyers_train = y_train > 0
mask_buyers_val   = y_val > 0

model_fs_reg = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.05, 
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1,
    early_stopping_rounds=30, eval_metric='rmse'
)

model_fs_reg.fit(
    X_train_mi[mask_buyers_train], y_train[mask_buyers_train],
    eval_set=[(X_val_mi[mask_buyers_val], y_val[mask_buyers_val])],
    verbose=False
)

imp_reg = pd.DataFrame({
    'feature': X_train_mi.columns,
    'importance': model_fs_reg.feature_importances_
}).sort_values(by='importance', ascending=False)

# Cumulative 95% cho Regression
cutoff_reg = (imp_reg['importance'].cumsum() <= 0.95).sum()
keep_reg = imp_reg.iloc[:cutoff_reg + 1]['feature'].tolist()

# --- 4C. GỘP (UNION) ĐẶC TRƯNG TỪ 2 TẦNG ---
final_features = list(set(keep_clf) | set(keep_reg))

print(f"   + Giữ lại từ Classifier: {len(keep_clf)} biến")
print(f"   + Giữ lại từ Regressor:  {len(keep_reg)} biến")
print(f"4. Sau Model Importance (Lấy Hợp - Union): Còn {len(final_features)} biến")

# ════════════════════════════════════════════════════════════════
# TỔNG KẾT VÀ LƯU TRỮ
# ════════════════════════════════════════════════════════════════

metadata_train_val = ['fullVisitorId', 'target_revenue', 'target_log_revenue', 'Fold', 'Split_Type']
metadata_test      = ['fullVisitorId', 'Fold', 'Split_Type']

train_df_final = train_df[metadata_train_val + final_features]
val_df_final   = val_df[metadata_train_val + final_features]
test_df_final  = test_df[metadata_test + final_features]

# Lưu pickle
train_df_final.to_pickle('../data/train_final.pkl')
val_df_final.to_pickle('../data/val_final.pkl')
test_df_final.to_pickle('../data/test_final.pkl')

features_dict = {
    'clf_features': keep_clf,
    'reg_features': keep_reg,
    'union_features': final_features
}
with open('../data/final_features.json', 'w') as f:
    json.dump(features_dict, f, indent=2)


print(f"\n=> HOÀN TẤT: Còn lại {len(final_features)} đặc trưng tinh túy nhất cho mô hình 2 tầng.")
print("Dữ liệu cuối cùng đã được lưu thành công.")

# In ra Top 10 của mỗi tầng để kiểm tra xem chúng có khác nhau nhiều không
print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (CLASSIFICATION) ---")
print(imp_clf.head(10))

print("\n--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (REGRESSION) ---")
print(imp_reg.head(10))


Số lượng đặc trưng ban đầu: 66
1. Sau Variance Threshold: Còn 62 biến
2. Sau Correlation Filter: Còn 44 biến
3. Đang thực hiện Mutual Information...
3. Sau Mutual Information (Top 95%): Còn 41 biến
4. Đang chạy XGBoost Importance cho cả 2 tầng (Classification + Regression)...
   + Giữ lại từ Classifier: 29 biến
   + Giữ lại từ Regressor:  34 biến
4. Sau Model Importance (Lấy Hợp - Union): Còn 39 biến

=> HOÀN TẤT: Còn lại 39 đặc trưng tinh túy nhất cho mô hình 2 tầng.
Dữ liệu cuối cùng đã được lưu thành công.

--- TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (CLASSIFICATION) ---
                             feature  importance
35                  hibernation_flag    0.150955
9      totals_transactionRevenue_sum    0.130262
15       geoNetwork_country_enc_mean    0.098756
38           interaction_hits_visits    0.092777
18  geoNetwork_subContinent_enc_mean    0.083409
13              device_isMobile_mean    0.046545
34               days_since_purchase    0.043169
24   device_operatingSystem_enc_me

Stage 1: Classification

In [1]:
from xgboost import XGBClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score, classification_report,
    confusion_matrix, fbeta_score, brier_score_loss
)
import numpy as np
import pandas as pd
import json

train_df = pd.read_pickle('../data/train_final.pkl')
val_df   = pd.read_pickle('../data/val_final.pkl')

with open('../data/final_features.json') as f:
    features_dict = json.load(f)

clf_features = features_dict['clf_features']
X_train = train_df[clf_features]
X_val   = val_df[clf_features]

y_train_buy = (train_df['target_revenue'] > 0).astype(int)
y_val_buy   = (val_df['target_revenue'] > 0).astype(int)

neg = (y_train_buy == 0).sum()
pos = (y_train_buy == 1).sum()

# ── SPW: sqrt cho PR-AUC tốt nhất từ thực nghiệm ──
spw = np.sqrt(neg / pos)  # sqrt tốt hơn cbrt theo thực nghiệm


print(f"Negative: {neg:,} | Positive: {pos:,} | SPW: {spw:.2f}x")

# ════════════════════════════════════════════════════════════
# TEMPORAL CALIBRATION SPLIT
# Dùng Fold sớm → train calibration
# Dùng Fold muộn → fit Isotonic
# Không shuffle → không temporal leakage
# ════════════════════════════════════════════════════════════
train_folds     = sorted(train_df['Fold'].unique())
n_cal_train     = int(len(train_folds) * 0.8)
cal_train_folds = train_folds[:n_cal_train]
cal_val_folds   = train_folds[n_cal_train:]

cal_val_mask = train_df['Fold'].isin(cal_val_folds)
X_cal_val    = X_train[cal_val_mask]
y_cal_val    = y_train_buy[cal_val_mask]

print(f"Cal train folds: {cal_train_folds} | Cal val folds: {cal_val_folds}")

# ════════════════════════════════════════════════════════════
# ENSEMBLE + ISOTONIC CALIBRATION
# ════════════════════════════════════════════════════════════
N_RUNS        = 5
p_cal_sum     = np.zeros(len(val_df))
pr_auc_list   = []
brier_list    = []

print(f"\nTraining {N_RUNS} classifiers...")

for seed in range(N_RUNS):
    clf = XGBClassifier(
        n_estimators=1000, max_depth=4,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=spw,
        eval_metric='aucpr', early_stopping_rounds=30,
        random_state=seed, n_jobs=-1
    )
    clf.fit(X_train, y_train_buy,
           eval_set=[(X_val, y_val_buy)], verbose=False)

    # Isotonic fit trên cal_val (fold muộn nhất)
    p_oof = clf.predict_proba(X_cal_val)[:, 1]
    iso   = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_oof, y_cal_val)

    p_raw = clf.predict_proba(X_val)[:, 1]
    p_cal = iso.predict(p_raw)
    p_cal_sum += p_cal

    pr  = average_precision_score(y_val_buy, p_cal)
    b   = brier_score_loss(y_val_buy, p_cal)
    pr_auc_list.append(pr)
    brier_list.append(b)
    print(f"  seed={seed} | PR-AUC={pr:.4f} | Brier={b:.6f} | iter={clf.best_iteration}")

p_avg          = p_cal_sum / N_RUNS
val_df['p_buy'] = p_avg

print(f"\nEnsemble PR-AUC: {average_precision_score(y_val_buy, p_avg):.4f}")
print(f"Ensemble Brier:  {brier_score_loss(y_val_buy, p_avg):.6f}")
print(f"Mean p_buy:      {p_avg.mean():.6f} (true rate: {y_val_buy.mean():.6f})")

# ════════════════════════════════════════════════════════════
# F2 THRESHOLD — Recall quan trọng hơn Precision
# Bỏ sót buyer ở Stage 1 → Stage 2 mất signal
# ════════════════════════════════════════════════════════════
thresholds = np.arange(0.001, 0.5, 0.001)
f2_scores  = [fbeta_score(y_val_buy, (p_avg >= t).astype(int), beta=2, zero_division=0)
               for t in thresholds]

best_t  = thresholds[np.argmax(f2_scores)]
best_f2 = max(f2_scores)

print(f"\nF2 Threshold: {best_t:.4f} | F2={best_f2:.4f}")

y_pred = (p_avg >= best_t).astype(int)
print(classification_report(y_val_buy, y_pred, digits=4))

cm = confusion_matrix(y_val_buy, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TP: {tp:,} ({tp/(tp+fn)*100:.1f}% buyers found)")
print(f"FN: {fn:,} ({fn/(tp+fn)*100:.1f}% buyers missed)")
print(f"\n✅ Saved: val_df['p_buy'] | best_t={best_t:.4f}")

Negative: 1,575,674 | Positive: 1,465 | SPW: 32.80x
Cal train folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | Cal val folds: [np.int64(5)]

Training 5 classifiers...
  seed=0 | PR-AUC=0.0619 | Brier=0.000718 | iter=63
  seed=1 | PR-AUC=0.0686 | Brier=0.000713 | iter=52
  seed=2 | PR-AUC=0.0625 | Brier=0.000716 | iter=86
  seed=3 | PR-AUC=0.0608 | Brier=0.000719 | iter=101
  seed=4 | PR-AUC=0.0645 | Brier=0.000712 | iter=68

Ensemble PR-AUC: 0.0742
Ensemble Brier:  0.000714
Mean p_buy:      0.000712 (true rate: 0.000744)

F2 Threshold: 0.0260 | F2=0.2055
              precision    recall  f1-score   support

           0     0.9996    0.9951    0.9973    336955
           1     0.0644    0.4542    0.1128       251

    accuracy                         0.9947    337206
   macro avg     0.5320    0.7246    0.5551    337206
weighted avg     0.9989    0.9947    0.9967    337206

TP: 114 (45.4% buyers found)
FN: 137 (54.6% buyers missed)

✅ Saved: val_df['p_buy'] | best_t=0.026

Stage 2: Regression

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

reg_features = features_dict['reg_features']

# ════════════════════════════════════════════════════════════
# 1. BUYERS ONLY
# ════════════════════════════════════════════════════════════
train_buyers = train_df[train_df['target_revenue'] > 0].copy()
val_buyers   = val_df[val_df['target_revenue'] > 0].copy()

X_train_b = train_buyers[reg_features]
X_val_b   = val_buyers[reg_features]
y_train_b = train_buyers['target_log_revenue']
y_val_b   = val_buyers['target_log_revenue']

p99_log = y_train_b.quantile(0.99)
print(f"Train buyers: {len(train_buyers):,} | p99_log: {p99_log:.4f}")

# Sample weight: boost high-value buyers, clip p99
weights_raw = np.expm1(y_train_b.values)
p99_w       = np.percentile(weights_raw, 99)
weights     = np.clip(weights_raw, 0, p99_w) / np.clip(weights_raw, 0, p99_w).mean()

print(f"Weights: min={weights.min():.2f} max={weights.max():.2f} mean={weights.mean():.2f}")

# ════════════════════════════════════════════════════════════
# 2. ENSEMBLE BUYERS REGRESSION
# ════════════════════════════════════════════════════════════
N_RUNS    = 5
e_sum_val = np.zeros(len(val_df))
e_sum_buy = np.zeros(len(val_buyers))
rmse_b_list = []

print(f"\nTraining {N_RUNS} buyer regressors...")

for seed in range(N_RUNS):
    reg_b = XGBRegressor(
        n_estimators=2000, max_depth=4,
        learning_rate=0.03, subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,   # tránh overfit tập buyers nhỏ

        reg_alpha=0.1, reg_lambda=1.0,  # regularization

        eval_metric='rmse', early_stopping_rounds=50,
        random_state=seed, n_jobs=-1
    )
    reg_b.fit(X_train_b, y_train_b,
              sample_weight=weights,
              eval_set=[(X_val_b, y_val_b)], verbose=False)

    pred_all = reg_b.predict(val_df[reg_features]).clip(0, p99_log)
    e_sum_val += pred_all
    e_sum_buy += pred_all[val_df['target_revenue'] > 0]

    rmse = np.sqrt(mean_squared_error(y_val_b, pred_all[val_df['target_revenue'] > 0]))
    rmse_b_list.append(rmse)
    print(f"  seed={seed} | RMSE buyers={rmse:.4f} | iter={reg_b.best_iteration}")

E_log_buyers = e_sum_val / N_RUNS

# ════════════════════════════════════════════════════════════
# 3. ENSEMBLE ALL-USERS REGRESSION (để compare)
# ════════════════════════════════════════════════════════════
e_sum_all = np.zeros(len(val_df))
rmse_all_list = []

print(f"\nTraining {N_RUNS} all-users regressors...")

for seed in range(N_RUNS):
    reg_all = XGBRegressor(
        n_estimators=2000, max_depth=4,
        learning_rate=0.03, subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,  # lớn hơn vì train all users

        reg_alpha=0.1, reg_lambda=1.0, # ban đầu là 1
        eval_metric='rmse', early_stopping_rounds=50,
        random_state=seed, n_jobs=-1
    )
    reg_all.fit(
        train_df[reg_features], train_df['target_log_revenue'],
        eval_set=[(val_df[reg_features], val_df['target_log_revenue'])],
        verbose=False
    )
    pred = reg_all.predict(val_df[reg_features]).clip(0)
    e_sum_all += pred
    rmse_all_list.append(np.sqrt(mean_squared_error(val_df['target_log_revenue'], pred)))
    print(f"  seed={seed} | RMSE all={rmse_all_list[-1]:.4f} | iter={reg_all.best_iteration}")

E_log_all = e_sum_all / N_RUNS

# ════════════════════════════════════════════════════════════
# 4. EVALUATE
# ════════════════════════════════════════════════════════════
baseline_b = np.sqrt(mean_squared_error(y_val_b, np.zeros(len(y_val_b))))

print(f"\n=== Buyers Regression ===")
print(f"RMSE mean: {np.mean(rmse_b_list):.4f} ± {np.std(rmse_b_list):.4f}")
print(f"Baseline:  {baseline_b:.4f}")

print(f"\n=== All-Users Regression ===")
print(f"RMSE mean: {np.mean(rmse_all_list):.4f} ± {np.std(rmse_all_list):.4f}")

val_df['e_log_revenue']     = E_log_buyers
val_df['e_log_revenue_all'] = E_log_all

print("\n✅ Saved: e_log_revenue | e_log_revenue_all | p99_log")

Train buyers: 1,465 | p99_log: 21.6405
Weights: min=0.00 max=10.77 mean=1.00

Training 5 buyer regressors...
  seed=0 | RMSE buyers=1.4068 | iter=1723
  seed=1 | RMSE buyers=1.4337 | iter=1311
  seed=2 | RMSE buyers=1.4411 | iter=1426
  seed=3 | RMSE buyers=1.4100 | iter=1770
  seed=4 | RMSE buyers=1.4249 | iter=1510

Training 5 all-users regressors...
  seed=0 | RMSE all=0.4876 | iter=2709
  seed=1 | RMSE all=0.4875 | iter=2738
  seed=2 | RMSE all=0.4875 | iter=2756
  seed=3 | RMSE all=0.4875 | iter=2999
  seed=4 | RMSE all=0.4875 | iter=2993

=== Buyers Regression ===
RMSE mean: 1.4233 ± 0.0132
Baseline:  18.0156

=== All-Users Regression ===
RMSE mean: 0.4875 ± 0.0000

✅ Saved: e_log_revenue | e_log_revenue_all | p99_log


Combine and evaluate

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

# ════════════════════════════════════════════════════════════════
# 1. LOAD VALUES
# ════════════════════════════════════════════════════════════════
P        = val_df['p_buy'].values
E_log    = val_df['e_log_revenue'].values
E_all    = val_df['e_log_revenue_all'].values
log_true = val_df['target_log_revenue'].values
y_true   = val_df['target_revenue'].values

buyers_mask = y_true > 0
baseline    = np.sqrt(mean_squared_error(log_true, np.zeros(len(log_true))))

# ════════════════════════════════════════════════════════════════
# 2. DUAL OUTPUT
# ════════════════════════════════════════════════════════════════

# --- OUTPUT 1: RANKING (Tối ưu RMSE, dùng để xếp hạng users) ---
pred_rank = (P * E_log).clip(0)

# --- OUTPUT 2: BUSINESS (Doanh thu kỳ vọng thực tế, dùng cho ROAS) ---
pred_biz = P * np.expm1(E_log)

# ════════════════════════════════════════════════════════════════
# 3. EVALUATION
# ════════════════════════════════════════════════════════════════
def eval_rmse(pred, name):
    pred_c  = np.clip(pred, 0, None)
    overall = np.sqrt(mean_squared_error(log_true, pred_c))
    b_rmse  = np.sqrt(mean_squared_error(log_true[buyers_mask], pred_c[buyers_mask]))
    nb_rmse = np.sqrt(mean_squared_error(log_true[~buyers_mask], pred_c[~buyers_mask]))
    beat    = '✅' if overall < baseline else '❌'
    print(f"{beat} {name:<40} RMSE={overall:.4f}  Buyers={b_rmse:.4f}  NonBuy={nb_rmse:.4f}")
    return overall

print(f"Baseline (toàn 0): {baseline:.4f}\n")

rmse_rank = eval_rmse(pred_rank,              "Ranking: P × E_log")
rmse_biz  = eval_rmse(np.log1p(pred_biz),     "Business: log1p(P × expm1(E_log))")
rmse_all  = eval_rmse(E_all,                  "Single Model: E_all only")

# ════════════════════════════════════════════════════════════════
# 4. SAVE DUAL OUTPUTS
# ════════════════════════════════════════════════════════════════
val_df['final_log_pred']     = pred_rank                  # Dùng cho Kaggle / RMSE
val_df['final_revenue_pred'] = np.expm1(pred_rank)        # Revenue từ Ranking
val_df['expected_revenue']   = pred_biz                   # Revenue kỳ vọng thực (cho ROAS)

# ════════════════════════════════════════════════════════════════
# 5. MARKETING METRICS (dựa trên Ranking Output)
# ════════════════════════════════════════════════════════════════
top_k   = int(len(val_df) * 0.10)
top_idx = np.argsort(pred_rank)[::-1][:top_k]

rev_capture  = y_true[top_idx].sum() / (y_true.sum() + 1e-9)
buyer_rate   = buyers_mask[top_idx].mean()
overall_rate = buyers_mask.mean()
lift         = buyer_rate / (overall_rate + 1e-9)

print(f"\n=== Marketing Metrics (Ranking Output) ===")
print(f"Revenue Capture@10%: {rev_capture:.4f}")
print(f"Buyer Rate@10%:      {buyer_rate:.4f}")
print(f"Lift@10%:            {lift:.2f}x")

# ════════════════════════════════════════════════════════════════
# 6. TOP 10 USERS — so sánh cả 2 output
# ════════════════════════════════════════════════════════════════
top10 = (
    val_df[['p_buy', 'e_log_revenue',
            'final_log_pred', 'final_revenue_pred',
            'expected_revenue',
            'target_revenue', 'target_log_revenue']]
    .assign(is_buyer=(val_df['target_revenue'] > 0).astype(int))
    .sort_values('final_log_pred', ascending=False)
    .head(10)
)

cols = ['final_revenue_pred', 'expected_revenue', 'target_revenue', 'p_buy', 'is_buyer']
top10 = top10[cols]

pd.set_option('display.float_format', '{:,.0f}'.format)
print(f"\n=== Top 10 Predicted CLV Users ===")
print(f"{'':>10} {'Ranking Rev':>15} {'Business Rev':>15} {'True Rev':>15} {'P(Buy)':>8} {'Buyer':>6}")
print("-" * 75)

for idx, row in top10.iterrows():
    print(f"{idx:>10} {row['final_revenue_pred']:>15,.0f} "
          f"{row['expected_revenue']:>15,.0f} "
          f"{row['target_revenue']:>15,.0f} "
          f"{row['p_buy']:>8.4f} "
          f"{int(row['is_buyer']):>6}")

print(f"\nTrue buyers in top10: {top10['is_buyer'].sum()}/10")

print("\n✅ Saved: final_log_pred | final_revenue_pred | expected_revenue")


Baseline (toàn 0): 0.4915

✅ Ranking: P × E_log                       RMSE=0.4804  Buyers=17.2624  NonBuy=0.0947
❌ Business: log1p(P × expm1(E_log))        RMSE=8.1370  Buyers=4.2583  NonBuy=8.1392
✅ Single Model: E_all only                 RMSE=0.4875  Buyers=17.8513  NonBuy=0.0218

=== Marketing Metrics (Ranking Output) ===
Revenue Capture@10%: 0.8940
Buyer Rate@10%:      0.0068
Lift@10%:            9.12x

=== Top 10 Predicted CLV Users ===
               Ranking Rev    Business Rev        True Rev   P(Buy)  Buyer
---------------------------------------------------------------------------
   1643138      84,319,064   2,110,231,821     330,690,000   0.8433      1
   1869041         115,295     351,522,031               0   0.5762      0
   1779906          76,646     170,121,789     622,440,000   0.5767      1
   1809032          24,803     275,627,189     324,420,000   0.5029      1
   1606177           7,668     526,162,467     461,940,000   0.4274      1
   1617004           4,697 